In [1]:
from transformers import CanineModel, CanineTokenizer
from classes.single_encoder import SingleEncoder
from classes.conversational_dataset import ConversationDataset
from classes.dual_encoder_single import DualEncoderAsSingle
from helpers.load_dual_encoder import load_dual_encoder_model
from helpers.load_single_encoder import load_single_encoder_model
from torch.utils.data import DataLoader
from torch.nn.functional import normalize
import torch.nn.functional as F
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import emoji
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import string
import os
import json
import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu118 for torchao version 0.15.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0117 21:20:11.145000 23008 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
# =============================
# SANITY CHECK: Embedding Norms
# =============================
# Purpose: Ensure embeddings are not collapsed and L2 normalization works.
# - mean/std check tells you if embeddings vary
# - norm check ensures embeddings are unit vectors

def evaluate_embedding_sanity(model, dataloader, device):
    model.eval()
    all_embeddings = []

    with torch.no_grad():
        for batch in dataloader:
            z = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )
            all_embeddings.append(z.cpu())

    embeddings = torch.cat(all_embeddings)

    return {
        "mean": embeddings.mean().item(),
        "std": embeddings.std().item(),
        "l2_norm_mean": embeddings.norm(dim=1).mean().item()
    }, embeddings


In [3]:
# =============================
# TOP-K RETRIEVAL (BY AUTHOR)
# =============================
# Purpose: Check if embeddings cluster messages from the same author.
# Method:
# 1. Compute cosine similarity between all embeddings
# 2. For each message, check if top-k nearest neighbors include the same author
# 3. Compute proportion (accuracy)

def evaluate_topk_author_accuracy(embeddings, dataloader, k=5):
    embeddings = normalize(embeddings, dim=1)
    sims = embeddings @ embeddings.T
    sims.fill_diagonal_(-1)

    authors = [item["author"] for item in dataloader.dataset]

    topk = sims.topk(k, dim=1).indices
    correct = 0

    for i, idxs in enumerate(topk):
        if authors[i] in [authors[j] for j in idxs]:
            correct += 1

    return correct / len(authors)


In [4]:
# =============================
# CLUSTER VISUALIZATION (t-SNE)
# =============================
# Purpose: Visual check if messages with the same author or style form clusters
# Method: Reduce embeddings to 2D using t-SNE and plot

def save_tsne_plot(embeddings, authors, out_path):
    z2d = TSNE(n_components=2, random_state=42).fit_transform(embeddings.numpy())
    colors = [hash(a) % 100 for a in authors]

    plt.figure(figsize=(8,6))
    plt.scatter(z2d[:,0], z2d[:,1], c=colors, cmap="tab20", s=5)
    plt.title("t-SNE of Style Discovery Loss (SDL)")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()


In [5]:
# =============================
# ABLATION TEST: Emoji removal
# =============================
# Purpose: Check if embeddings encode emojis
# Method: Remove emojis from content and compare embedding shift

# --------------------------------------------------
# Assumptions:
# - model is already loaded and trained
# - tokenizer is already loaded
# - val_loader is built from ConversationDataset
# - ConversationDataset returns:
#   input_ids, attention_mask, author, content
# --------------------------------------------------


def run_style_ablation(
    model, tokenizer, texts, ablation_fn, device, batch_size=8
):
    def embed(texts):
        zs = []
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(
                texts[i:i+batch_size],
                padding=True,
                truncation=True,
                return_tensors="pt"
            )
            with torch.no_grad():
                z = model(
                    enc["input_ids"].to(device),
                    enc["attention_mask"].to(device)
                )
                zs.append(normalize(z, dim=1).cpu())
        return torch.cat(zs)

    z_orig = embed(texts)
    z_mod = embed([ablation_fn(t) for t in texts])

    shift = 1 - torch.nn.functional.cosine_similarity(z_orig, z_mod, dim=1)

    return {
        "mean": shift.mean().item(),
        "std": shift.std().item(),
        "min": shift.min().item(),
        "max": shift.max().item()
    }


In [6]:
@torch.no_grad()
def retrieval_self_consistency_score(
    embeddings: torch.Tensor,
    k: int = 5,
    batch_size: int = 1024
) -> float:
    """
    Retrieval Self-Consistency Score (RSCS)

    Args:
        embeddings: Tensor (N, D), NOT assumed normalized
        k: number of nearest neighbors
        batch_size: batch size for similarity computation

    Returns:
        RSCS score (float), higher = better
    """
    # Ensure cosine similarity
    embeddings = F.normalize(embeddings, dim=1)

    N, D = embeddings.shape
    device = embeddings.device

    scores = []

    for start in range(0, N, batch_size):
        end = min(start + batch_size, N)
        batch = embeddings[start:end]                 # (B, D)

        # Cosine similarity to all embeddings
        sim = batch @ embeddings.T                    # (B, N)

        # Remove self similarity
        rows = torch.arange(start, end, device=device)
        sim[torch.arange(end - start), rows] = -float("inf")

        # Top-k retrieval
        topk_sim, topk_idx = torch.topk(sim, k=k, dim=1)  # (B, k)

        # ---- Term 1: query-to-neighbor relevance ----
        relevance = topk_sim.mean(dim=1)               # (B,)

        # ---- Term 2: neighbor-to-neighbor coherence ----
        neigh_emb = embeddings[topk_idx]               # (B, k, D)
        pairwise = torch.einsum("bkd,bqd->bkq", neigh_emb, neigh_emb)

        # Upper triangle without diagonal
        iu = torch.triu_indices(k, k, offset=1)
        coherence = pairwise[:, iu[0], iu[1]].mean(dim=1)

        scores.append(relevance * coherence)


    return torch.cat(scores).mean().item()


In [7]:
def run_full_evaluation(
    model,
    tokenizer,
    val_loader,
    output_dir,
    device="cuda"
):
    print("Starting full evaluation...")
    os.makedirs(output_dir, exist_ok=True)
    model.to(device)

    texts = [item["content"] for item in val_loader.dataset]
    authors = [item["author"] for item in val_loader.dataset]

    results = {}

    # 1. Sanity
    print("Evaluating embedding sanity...")
    sanity, embeddings = evaluate_embedding_sanity(model, val_loader, device)
    results["embedding_sanity"] = sanity

    # 2. Retrieval
    print("Evaluating top-5 author accuracy...")
    results["top5_author_accuracy"] = evaluate_topk_author_accuracy(
        embeddings, val_loader, k=5
    )

    # 3. Retrieval Self-Consistency Score
    print("Evaluating retrieval self-consistency score...")
    results["retrieval_self_consistency_score"] = retrieval_self_consistency_score(embeddings.to(device), k=5),


    # 4. Ablations
    print("Evaluating ablations...")
    results["ablations"] = {
        "emoji": run_style_ablation(
            model, tokenizer, texts,
            lambda t: emoji.replace_emoji(t, ""),
            device
        ),
        "punctuation": run_style_ablation(
            model, tokenizer, texts,
            lambda t: t.translate(str.maketrans("", "", string.punctuation)),
            device
        ),
        "lowercase": run_style_ablation(
            model, tokenizer, texts,
            lambda t: t.lower(),
            device
        )
    }

    # 5. Save t-SNE
    print("Saving t-SNE plot...")
    save_tsne_plot(
        embeddings, authors,
        os.path.join(output_dir, "tsne.png")
    )

    # 6. Save JSON
    with open(os.path.join(output_dir, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

    return results


In [8]:
def rscs_eval(
    model,
    tokenizer,
    val_loader,
    output_dir,
    device="cuda",
    max_k=10000,
    step=500
):
    print("Starting full evaluation for multiple k values...")
    os.makedirs(output_dir, exist_ok=True)
    model.to(device)

    # 1. Sanity
    print("Evaluating embedding sanity...")
    sanity, embeddings = evaluate_embedding_sanity(model, val_loader, device)

    # 2. Compute RSCS for k=5,10,...,max_k
    ks = list(range(step, max_k + 1, step))
    rscs_scores = []

    print("Evaluating retrieval self-consistency score for multiple k values...")
    embeddings = embeddings.to(device)
    for k in ks:
        score = retrieval_self_consistency_score(embeddings, k=k)
        print(f"k={k}: RSCS={score:.4f}")
        rscs_scores.append(score)

    # 3. Save results to JSON
    results = {"retrieval_self_consistency_score": {str(k): score for k, score in zip(ks, rscs_scores)}}
    with open(os.path.join(output_dir, "rscs_results.json"), "w") as f:
        json.dump(results, f, indent=2)

    # 4. Plot graph
    plt.figure(figsize=(8, 5))
    plt.plot(ks, rscs_scores, marker='o')
    plt.title("Retrieval Self-Consistency Score vs k")
    plt.xlabel("k")
    plt.ylabel("RSCS")
    plt.ylim(-0.05, 1.05)
    plt.grid(True)
    plt.xticks(ks)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "rscs_plot.png"))
    plt.show()

    return results


In [9]:
# model_name = "author_contrastive"
# validation_data = "retriever_val_undersampled"

# # Load model
# save_dir = "../output/models/" + model_name
# # Load model
# model, tokenizer, device = load_single_encoder_model(save_dir)
# # model, tokenizer, device = load_dual_encoder_model(save_dir)
# # model = DualEncoderAsSingle(model)

# # Load validation data
# rows = pd.read_csv("../data/train/" + validation_data + ".csv").to_dict(orient="records")
# val_dataset = ConversationDataset(rows, tokenizer, max_length=512)

# val_loader = DataLoader(
#     val_dataset,
#     batch_size=32,
#     shuffle=False,
#     num_workers=4
# )

# # Run evaluation
# results = run_full_evaluation(
#     model,
#     tokenizer,
#     val_loader,
#     output_dir="../output/evaluations/" + model_name,
#     device=device
# )


In [10]:
model_name = "style_discovery_loss"
validation_data = "retriever_val_undersampled"

# Load model
save_dir = "../output/models/" + model_name
# Load model
# model, tokenizer, device = load_single_encoder_model(save_dir)
model, tokenizer, device = load_dual_encoder_model(save_dir)
model = DualEncoderAsSingle(model)

# Load validation data
rows = pd.read_csv("../data/train/" + validation_data + ".csv").to_dict(orient="records")
val_dataset = ConversationDataset(rows, tokenizer, max_length=512)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

# Run evaluation
run_full_evaluation(
    model,
    tokenizer,
    val_loader,
    output_dir="../output/evaluations/" + model_name,
    device=device
)

# rscs_eval(
#     model,
#     tokenizer,
#     val_loader,
#     output_dir="../output/evaluations/" + model_name,
#     device=device
# )


Loaded model weights from ../output/models/style_discovery_loss\dual_encoder.pt
Loaded projection heads from ../output/models/style_discovery_loss\projection_head.pt
Loaded tokenizer from ../output/models/style_discovery_loss
Model loaded successfully
Device: cuda
Model dir: ../output/models/style_discovery_loss
Starting full evaluation...
Evaluating embedding sanity...
Evaluating top-5 author accuracy...
Evaluating retrieval self-consistency score...
Evaluating ablations...
Saving t-SNE plot...


{'embedding_sanity': {'mean': 0.012448360212147236,
  'std': 0.12437885254621506,
  'l2_norm_mean': 1.0},
 'top5_author_accuracy': 0.13275,
 'retrieval_self_consistency_score': (0.9999377727508545,),
 'ablations': {'emoji': {'mean': 0.002428583102300763,
   'std': 0.0552574098110199,
   'min': -2.384185791015625e-07,
   'max': 1.3849544525146484},
  'punctuation': {'mean': 0.06265062838792801,
   'std': 0.274896502494812,
   'min': -2.384185791015625e-07,
   'max': 1.3913706541061401},
  'lowercase': {'mean': 0.10136131942272186,
   'std': 0.3476560413837433,
   'min': -2.384185791015625e-07,
   'max': 1.392000436782837}}}

In [ ]:
model_name = "google/canine-s"  # base CANINE model
validation_data = "retriever_val_undersampled"

# Load tokenizer and encoder
tokenizer = CanineTokenizer.from_pretrained(model_name)
encoder = CanineModel.from_pretrained(model_name)

# Wrap in StyleEncoder
model = SingleEncoder()
model.encoder = encoder  # replace the encoder
model.to(device)
model.eval()

# Load validation data
rows = pd.read_csv(f"../data/train/{validation_data}.csv").to_dict(orient="records")
val_dataset = ConversationDataset(rows, tokenizer, max_length=512)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

# Run evaluation
run_full_evaluation(
    model,
    tokenizer,
    val_loader,
    output_dir=f"../output/evaluations/base_canine",
    device=device
)

# rscs_eval(
#     model,
#     tokenizer,
#     val_loader,
#     output_dir="../output/evaluations/" + model_name,
#     device=device
# )


Starting full evaluation...
Evaluating embedding sanity...
Evaluating top-5 author accuracy...
Evaluating retrieval self-consistency score...
Evaluating ablations...
Saving t-SNE plot...


{'embedding_sanity': {'mean': -0.006101893726736307,
  'std': 0.08817756175994873,
  'l2_norm_mean': 1.0},
 'top5_author_accuracy': 0.21,
 'retrieval_self_consistency_score': (0.7398474216461182,),
 'ablations': {'emoji': {'mean': 0.0020076208747923374,
   'std': 0.019761255010962486,
   'min': -2.384185791015625e-07,
   'max': 0.46909254789352417},
  'punctuation': {'mean': 0.044543471187353134,
   'std': 0.06983083486557007,
   'min': -2.384185791015625e-07,
   'max': 0.729163408279419},
  'lowercase': {'mean': 0.05745788663625717,
   'std': 0.07136289775371552,
   'min': -2.384185791015625e-07,
   'max': 0.4231099486351013}}}